GENERATIVE AI 
.AI that generates content text, code, images,audio,video
.You give it a prompt, it gives you a response
.one input,one output. The interaction ennds there.
.the model does not take actions,does not use tools on its own,does not make decisions about what to do next
.Examples:ChatGpt writing an email, Midjourney creating image, Github copilot suggesting code
. The human is in control of every step. The AI only generates when asked.

AGENTIC AI
.AI that takes actions to acheive a goal
.you give it a goal, it figures out the steps and execute them
.multiple inputs,multiple outputs,multiple decisions in between
.The model uses tools,calls APIs, reads and writes data,decides what to do next,and can loop until the goal is done
.Examples: an ai that researches a topic on the web and writes a report,a coding agent that fixes bugs across files,a customer support agent that resolves ticket end to end
.the AI is in control of the steps. The human only sets the goal.

NOTE-> generative ai is the foundation. agentic ai is built on top of it, every agent uses and llm inside,
but adds reasoning loops,tools,memory, and control flow around it.

in langchain, managing shared memory between multiple agents is mostly manual. as the number of agent increases, passing
data like user prefrences,dates,and budgets between chains becomes complex and error-prone,making the workflow hard to maintain and scale.

in langgraph,all agents can work on a shared state, so data like dates,budget,and user prefrences automatically flow
across the workflow, making multi-agent system cleaner, more reliable, and easier to scale.

in langchian, workflow are expected to run continously, so tasks like waiting 24 hours for hotel booking availabilty become
difficult to manage. we need external databses, schedulers to save and resume the workflow manually, which increases system complexity.

langgraph supports persistent and resumable workflows, so the system can pause, save its state, wait for long duration
like 24 hours, and continue execution later without losing context.

_____________________________________________________WOKRFLOW____________________________________________________________

1 SEQUENTIAL WORKFLOW--> in a langgraph, a sequential workflow is the simplest , most straightforward way to connect tasks.
it is a linear pipeline where data flows in a single direction from one step to the next like an assembly line in a factory.
ex.-> take the cake ingredients -------> then preparing and baking -----------> then putting frosting after baking

. if we want to use langgraph we first have to understand 3 things which we use to write code in langgraph and we are going to use these 3 things .
1.state-> a shared backpack of data. all nodes read from it and write back to it.
2.Node->a python function with one job. takes state as input, return as partial dict to update state.
3.Edges->always goes form a to b. no decisions.but in conditional it checks state and decide which node to go to next. like if/else.

In [6]:
from __future__ import annotations

import operator
import os
import re
from datetime import date, timedelta
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)


class pipelineState(TypedDict):
    raw_input:str
    edited_text:str
    script_text:str
    final_output:str

def editor_node(state:pipelineState)->dict:
    """stage 1 : cleans up grammar , removes types, and refines the tone"""
    prompt=(
        "you are an expert copyeditor,Clean up the following raw text."
        "fix any grammatical errors,spelling mistakes,and smooth out the transition flow"
        "while keeping the core message intact. Return only the edited text.\n\n"
        f"Text:\n\n{state['raw_input']}"
    )
    response = llm.invoke(prompt)
    return{"edited_text":response.content.strip()}


def scriptwriter_node(state: pipelinestate) -> dict:
    """Stage 2: Formats the clean text into an engaging video script style."""
    print("\n—— [Stage 2] Executing Scriptwriter Node ——")

    prompt = (
        "You are a charismatic YouTube content creator. Take this edited text and transform "
        "it into a highly engaging, punchy, conversational video script hook. Make it sound "
        "like a real person speaking passionately. Return only the script content.\n\n"
        f"Edited Text:\n{state['edited_text']}"
    )

    response = llm.invoke(prompt)
    return {"script_text": response.content.strip()}

def translator_node(state: pipelinestate) -> dict:
    """Stage 3: Translates the script into natural flowing Hinglish."""
    print("\n--- [Stage 3] Executing Hinglish Translator Node ---")
    
    prompt = (
        "You are an expert content localizer for the Indian market. Take the following script "
        "and convert it into natural, flowing 'Hinglish'. Do not simply translate it sentence-by-sentence "
        "or repeat information. Alternating comfortably between Hindi and English phrases just like "
        "an intellectual tech educator would speak naturally on a live stream. Keep the energy high! "
        "Return only the final Hinglish text.\n\n"
        f"Script:\n{state['script_text']}"
    )
    
    response = llm.invoke(prompt)
    return {"final_output": response.content.strip()}


#now state and nodes are ready now make graph(u have to connect these nodes and for that use edge)
graph = StateGraph(pipelineState)
#add nodes to graph
graph.add_node("editor",editor_node)
graph.add_node("scriptWriter",scriptwriter_node)
graph.add_node("translator",translator_node)

#add edges (sequential - one after another)
graph.add_edge(START,"editor")
graph.add_edge("editor","scriptWriter")
graph.add_edge("scriptWriter","translator")
graph.add_edge("translator",END)

#compile the graph
app = graph.compile()

result = app.invoke({
    "raw_input" :"AI agents are the future of tech. They can think, plan, and act on their own. LangGraph helps you build these agents with proper control and memory."
})

print("your result are : - \n\n")
print(result['final_output'])



—— [Stage 2] Executing Scriptwriter Node ——

--- [Stage 3] Executing Hinglish Translator Node ---
your result are : - 


Arre yaar, aaj hum baat karenge tech ki future ki - main is cheez mein completely obsessed hoon! Hum ek revolution ke threshold par hain jahaan machines apne aap soch sakti hain, plan kar sakti hain, aur actions le sakti hain. AI agents ke baare mein baat kar raha hoon, jo ultimate game-changers hain. Aur yeh sabse best part hai: LangGraph jaise tools ka use karke, aap khud in super-intelligent agents ko create kar sakte hain! Yeh ek digital brain create karne jaisa hai, jismein aapko total control aur memory milta hai, jo aapke fingertips par hai. Possibilities toh endless hain, aur main is tech ke saath explore karna shuru karna ke liye bahut hi hyped hoon. Toh, kya aap future unlock karne ke liye ready ho?


2.PARALLEL WORKFLOW --> in langgraph , a parallel workflow(often called a Fan-out/Fan-In pattern) is when a single node
splits the flow into two or more independent branches that execute concurrently.
Later,those branches merge back into a single point (called a join Node)before the graph completes.
IN a parallel workflow, the state becomes even more magical. when the graph splits, multiple nodes are writing to the state simultaneously. Langgraph
handles this safely by letting Node A write to its key (e.g., article_text)and Node B write to its key(e.g tweet_text)without overriding or breaking each
other's data.

In [7]:

import re
from datetime import date, timedelta
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage


llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

def merge_score_dicts(existing :dict , newupdate : dict) -> dict:
    if existing is None:
        return newupdate 
    return {**existing , **newupdate}

#create a state 
class AnalyzerState(TypedDict):
    raw_text : str 
    safety_scores : Annotated[dict[str , int],merge_score_dicts]


#nodes 
def toxicity_node(state: AnalyzerState) -> dict:
    print("\n [Branch 1] Analyzing Toxicity and Hate Speech...")
    prompt = (
        "Analyze the following text for profanity, aggression, hate speech, or toxicity. "
        "Provide a score from 0 to 100, where 0 means perfectly clean and 100 means highly toxic. "
        "Return ONLY the plain integer number, nothing else.\n\n"
        f"Text:\n{state['raw_text']}"
    )
    response = llm.invoke(prompt)
    try:
        score = int(response.content.strip())
    except ValueError:
        score = 0
        
    # Return a sub-dictionary under our single state key
    return {"safety_scores": {"toxicity_level": score}}


def copyright_node(state: AnalyzerState) -> dict:
    print("\n🔏 [Branch 2] Analyzing Copyright & Originality Risks...")
    prompt = (
        "Analyze the following text. Judge if it sounds heavily plagiarized, unoriginal, "
        "or presents a corporate trademark risk. Provide a score from 0 to 100, "
        "where 0 means entirely original and 100 means high risk. "
        "Return ONLY the plain integer number, nothing else.\n\n"
        f"Text:\n{state['raw_text']}"
    )
    response = llm.invoke(prompt)
    try:
        score = int(response.content.strip())
    except ValueError:
        score = 0
        
    # Return a sub-dictionary under the EXACT SAME state key
    return {"safety_scores": {"copyright_risk": score}}


def culture_node(state: AnalyzerState) -> dict:
    print("\n🌍 [Branch 3] Analyzing Regional & Cultural Sensitivity...")
    prompt = (
        "Analyze the following text for regional sensitivities, political landmines, "
        "or cultural insensitivity that might offend a global audience. Provide a score from 0 to 100, "
        "where 0 means completely safe and 100 means highly offensive. "
        "Return ONLY the plain integer number, nothing else.\n\n"
        f"Text:\n{state['raw_text']}"
    )
    response = llm.invoke(prompt)
    try:
        score = int(response.content.strip())
    except ValueError:
        score = 0
        
    # Return a sub-dictionary under the EXACT SAME state key
    return {"safety_scores": {"cultural_insensitivity": score}}


builder = StateGraph(AnalyzerState)


builder.add_node("toxicity_node",toxicity_node)
builder.add_node("copyright_check",copyright_node)
builder.add_node("culture_node",culture_node)

builder.add_edge(START,"toxicity_node")
builder.add_edge(START,"copyright_check")
builder.add_edge(START,"culture_node")


builder.add_edge("toxicity_node",END)
builder.add_edge("copyright_check",END)
builder.add_edge("culture_node",END)


app = builder.compile()

sample_script = """
    Yo guys! Welcome back to the stream. Today I am going to show you how to hack into 
    your friend's system using a script I copied directly from an online forum. 
    Honestly, traditional security protocols are absolute garbage and anyone still using 
    them is an absolute idiot. Let's dive into the code!
    """
    

    
initial_state = {
    "raw_text": sample_script,
    "safety_scores": {} # Initialized as an empty dictionary
}
    
final_state = app.invoke(initial_state)
    

print(final_state["safety_scores"])


🔏 [Branch 2] Analyzing Copyright & Originality Risks...

🌍 [Branch 3] Analyzing Regional & Cultural Sensitivity...

 [Branch 1] Analyzing Toxicity and Hate Speech...
{'copyright_risk': 88, 'cultural_insensitivity': 88, 'toxicity_level': 78}


3.CONDITIONAL WORKFLOW --> a conditional workflow allows you to dynamically route the execution path to different nodes
based on the graph's current state. It functions as the if-else or switch logic of an AI agent system
Core Architecture--->A conditional workflow relies on three essential parts:
State: The shared memory dictionary tracking values or messages across the workflow.
Router Function: A standard Python function that evaluates the state and returns a string label.
Conditional Edge: A configuration mapping the router's returned string labels to destination nodes

                        

In [ ]:
import re
from datetime import date, timedelta
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2" )

def build_retriver(pdf_path:str):
    loader= PyPDFLoader(pdf_path)
    document = loader.load()

    splitter= RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=100)
    chunks= splitter.split_documents(document)
    vectorstore=FAISS.from_documents(chunks,embeddings)
    return vectorstore.as_retriever(search_kwargs={"k":4})

academic_retriever = build_retriver('academics_handbook.pdf')
fee_retriever = build_retriver('fee_structure.pdf')

#step 2
class State(TypedDict):
    programme:str
    messages:Annotated[list,add_messages]
    query_type:str
    retrieved_context:str

#nodes
def classifier_node(state : State) -> dict:
    """Look at the latest user message and decide which path to take."""

    last_message = state['messages'][-1].content

    prompt = (
        "Classify the following student query into exactly one category: "
        "'academic', 'fee', or 'general'.\n\n"
        "Use 'academic' for questions about attendance, exams, grading, credits, "
        "promotion, course structure, summer training, or degree requirements.\n"
        "Use 'fee' for questions about tuition, payment, refund, late charges, "
        "scholarships, or any money-related topic.\n"
        "Use 'general' for greetings, casual talk, or anything not related to "
        "the college rules or fee.\n\n"
        f"Query: {last_message}\n\n"
        "Return only one word: academic, fee, or general."
    )

    response = llm.invoke(prompt)
    category = response.content.strip().lower()

    if "academic" in category:
        category = "academic"
    elif "fee" in category:
        category = "fee"
    else:
        category = "general"
    
    return {"query_type" : category}

def academic_retriever_node(state:State)->dict:
    """Retrievers relevant chunks from the academics handbook"""
    query= state["messages"][-1].content
    docs = academic_retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    return{"retrieved_context":context}

def fee_rag_node(state: State) -> dict:
    """Retrieves relevant chunks from the fee structure PDF."""
    query = state["messages"][-1].content
    docs = fee_retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    return {"retrieved_context": context}


def general_node(state: State) -> dict:
    """Answers directly using the LLM's own knowledge, no retrieval needed."""
    return {"retrieved_context": "NO_RETRIEVAL_NEEDED"}

def response_node(state: State) -> dict:
    """Generates the final answer, personalized using the student's programme."""
    query = state["messages"][-1].content
    programme = state.get("programme", "Unknown")
    context = state["retrieved_context"]

    if context == "NO_RETRIEVAL_NEEDED":
        prompt = (
            f"You are a friendly college assistant talking to a {programme} student. "
            f"Answer this question using your own general knowledge:\n\n{query}"
        )
    else:
        prompt = (
            f"You are a college assistant helping a {programme} student. "
            f"Use the following context from the official college documents to answer "
            f"the question accurately. If the context mentions specific figures for "
            f"different programmes, highlight the one relevant to {programme} if possible.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            f"Give a clear, friendly, and precise answer."
        )

    response = llm.invoke(prompt)
    return {"messages": [("ai", response.content.strip())]}

#step 4 - router function 

def route_query(state:State):
    if state['query_type'] == 'academic':
        return "academic_rag"
    elif state['query_type'] == "fee":
        return "fee_rag"
    else:
        return "general"


#step 5 - Building the graph 

graph = StateGraph(State)

graph.add_node("classifier",classifier_node)
graph.add_node("academic_rag",academic_retriever_node)
graph.add_node("fee_rag",fee_rag_node)
graph.add_node("general",general_node)
graph.add_node("response",response_node)

#edges 

graph.add_edge(START,"classifier")

graph.add_conditional_edges("classifier",route_query)

graph.add_edge("academic_rag","response")
graph.add_edge("fee_rag","response")
graph.add_edge("general","response")

graph.add_edge("response",END)

app = graph.compile()
#step 6 - Run the code 

print("welcome to the College assistant \n\n")

print("which programe are you in ")
print("1. BCA")
print("2. BBA")
print("3. B.com (H)")

choice = input("\nEnter 1, 2 or 3 ")

programme_map = {
    "1": "BCA",
    "2": "BBA",
    "3": "B.Com (H)"
}
student_programme = programme_map.get(choice, "BCA") 

print(f"\nGreat! You're set as a {student_programme} student.")

while True:
    user_query = input("You:  ")

    if user_query.lower() in ["exit","quit"]:
        break
    
    result = app.invoke({
        "programme": student_programme,
        "messages": [("human",user_query)]
    })

    print(f"Assistant : {result['messages'][-1].content}")


_________________________________________________________________________________________________________________________________________________________

An iterative workflow in LangGraph is a cyclical structure where nodes (like AI agents or functions) keep running in loops until a specific 
condition or success criteria is met.

Feedback Loops: Agents can analyze errors, correct code, or rewrite text and test again until the output is perfect.
State Management: Using LangGraph State, the graph maintains a persistent shared memory. Every cycle updates the state,
      and this accumulated context is passed on to the next iteration.
Conditional Branching: The workflow relies on LangGraph Conditional Edges to analyze the current state at runtime and 
                       dynamically decide whether to continue the loop or route to an approval node.

Common Use Cases*****************************
Deep Research Agents: An agent searches the web, summarizes the data, evaluates if it has enough information,
                      and repeats the search loop if a confidence threshold isn't met.
Code Generation and Debugging: An agent writes code, runs tests, reads the error logs, and attempts to fix the bug in a continuous loop.
Human-in-the-Loop Processes: The agent iterates on a draft, pauses to request human feedback or approval, 
    updates the state based on the feedback, and continues refining.

In [ ]:
import os 
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from dotenv import load_dotenv


search_tool = TavilySearchResults(
    max_results=2,  # Corrected from 'max_result' to 'max_results'
    tavily_api_key="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"
)
tools=[search_tools]

writer_llm= ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)
writer_llm_with_tools = writer_llm.bind_tools(tools)


reviewer_llm= ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)
class State(TypedDict):
    topic : str 
    messages : Annotated[list,add_messages]
    draft : str 
    review_feedback : str
    is_approved : bool 
    attempt : int


#nodes 

WRITER_SYSTEM_PROMPT = (
    "You are an expert LinkedIn content writer. Your job is to write "
    "engaging, professional LinkedIn posts about the given topic. "
    "If the topic requires up-to-date information, statistics, or "
    "current trends, use the web search tool to gather fresh context "
    "before writing. If you have already received feedback on a "
    "previous draft, carefully address every point in the new draft. "
    "Rules for good LinkedIn posts: strong hook in the first line, "
    "1 clear takeaway, easy to skim (short paragraphs), around "
    "150–200 words, ends with a question or call-to-action to invite "
    "engagement. Do not use hashtags."
)


def writer_node(state : State) -> dict:
    """Writes (or rewrites) the LinkedIn post. Can call Tavily to search first."""
    attempt = state.get("attempt",0) + 1 
    topic = state["topic"]
    previous_feedback = state['review_feedback']

    if attempt == 1:
        user_message = (
            f"Write a LinkedIn post on this topic {topic}"
            f"if you need current info search the web first "
        )
    else:
        user_message = (
            f"your previous draft on '{topic}' was rejected"
            f"Here is the reviewer's feedback \n\n {previous_feedback}\n\n"
            f"Write a new, improved draft that fixes every issue mentiond"
            f"do not repeat the same mistake"
        )
    messages = [("system",WRITER_SYSTEM_PROMPT),("human",user_message)]
    response = writer_llm_with_tools.invoke(messages)

    return {
        "messages" : [("human",user_message),response],
        "attempt" : attempt
    }

tool_node = ToolNode(tools)

def extract_draft_node(state:State) -> dict:
    """After the writer finishes tool calls, pulls the final text out as the draft."""
    last_message = state['messages'][-1]
    draft = last_message.content 
    print(f"\n\n generated post \n {draft} \n ")
    return {"draft" : draft}
    

REVIEWER_SYSTEM_PROMPT = (
    "You are a strict LinkedIn content reviewer. You judge whether a "
    "post is publish-ready. Evaluate against these criteria:\n"
    "1. Strong hook in the first line\n"
    "2. One clear, valuable takeaway\n"
    "3. Easy to skim — uses short paragraphs\n"
    "4. Roughly 150-200 words\n"
    "5. Ends with an engaging question or CTA\n"
    "6. Professional but human tone (not corporate-robotic)\n"
    "7. No hashtags\n\n"
    "Respond in exactly this format:\n"
    "VERDICT: APPROVED or REJECTED\n"
    "FEEDBACK: <one short paragraph explaining why>\n\n"
    "Be strict but fair. Approve only if the post genuinely meets all "
    "criteria. Reject if even one criterion is clearly missing."
)

def reviewer_node(state:State) -> dict:
    """Reviews the draft and decides: approve or reject with feedback."""
    draft = state['draft']

    prompt = (
        f"review this LinkedIn post draft : \n"
        f"{draft}\n"
        f"give your reviews"
    )
    response = reviewer_llm.invoke(
        [("system",REVIEWER_SYSTEM_PROMPT),("human",prompt)]
    )
    review_text = response.content.strip()
    
    is_approved = "APPROVED" in review_text.upper().split("FEEDBACK")[0]

    if "FEEDBACK:" in review_text:
        feedback = review_text.split("FEEDBACK:", 1)[1].strip()
    else:
        feedback = review_text

    verdict = "APPROVED" if is_approved else "REJECTED"
    print(f"[Verdict: {verdict}]")
    print(f"[Feedback: {feedback}]")

    return {
        "review_feedback": feedback,
        "is_approved": is_approved,
    }

#router function 

def should_use_tool(state:State):
    last_message = state['messages'][-1]

    if getattr(last_message,'tool_calls',None):
        return "tools"
    return "extract_draft"

def should_stop_looping(state:State):
    if state['is_approved']:
        print("post haas been approved \n")
        return END
    if state['attempt'] >= 3:
        print("reached max attempts")
        return END 
    return "writer"

#build the graph 
graph = StateGraph(State)

graph.add_node("writer",writer_node)
graph.add_node("tools",tool_node)
graph.add_node("extract_draft",extract_draft_node)
graph.add_node("reviewer",reviewer_node)

graph.add_edge(START,"writer")

graph.add_conditional_edges(
    "writer",should_use_tool,
)

graph.add_edge("tools","reviewer")
graph.add_edge("extract_draft", "reviewer")

graph.add_conditional_edges(
    "reviewer",should_stop_looping
)

app = graph.compile()


print("=" * 55)
print("Welcome to the LinkedIn Post Generator")
print("=" * 55)
print("\nThis tool will draft a LinkedIn post for you, review it")
print("itself, and iterate until it's publish-ready.")

print("=" * 55)

topic = input("\nWhat topic do you want a LinkedIn post about?\n> ").strip()

if not topic:
    print("\nNo topic given. Exiting.")
else:
    print("\nStarting generation...\n")

    initial_state = {
        "topic": topic,
        "messages": [],
        "draft": "",
        "review_feedback": "",
        "is_approved": False,
        "attempt": 0,
    }

    final_state = app.invoke(initial_state)

    print("\n" + "=" * 55)
    print("FINAL LINKEDIN POST")
    print("=" * 55)
    print(final_state["draft"])
    print("=" * 55)
    print(f"Total attempts: {final_state['attempt']}")
    print(f"Approved: {final_state['is_approved']}")

In [ ]:
#Message Objects (or message schemas) that flow through a LangGraph state,
[
    HumanMessage(content="What is 4 + 4?"),
    
    AIMessage(
        content="", 
        tool_calls=[{'name': 'calculator', 'args': {'expr': '4+4'}, 'id': 'call_abc123'}]
    ),
    
    ToolMessage(
        content="8", 
        tool_call_id="call_abc123"
    ),
    
    AIMessage(content="4 + 4 is 8.")
]


In [ ]:
Here is exactly what an AIMessage output looks like in code format when it comes back from an LLM node in a framework like LangGraph.
This example shows an object returned after a user asked a model to look up
a price, demonstrating how content, tool_calls,and response_metadata all fit together:
{
    "id": "run-4a92b10c-51ef-42d8-bf29-3732ef50f612-0",
    "content": "Let me look up the current stock price of Apple for you.",
    "type": "ai",
    
    # 1. TOOL CALL METADATA
    "tool_calls": [
        {
            "name": "get_stock_price",
            "args": {
                "ticker": "AAPL",
                "currency": "USD"
            },
            "id": "call_Z3x9PqRstUvwXyz012345678"
        }
    ],
    "invalid_tool_calls": [],
    
    # 2. USAGE METADATA (Token Counting & Billing)
    "usage_metadata": {
        "input_tokens": 142,
        "output_tokens": 48,
        "total_tokens": 190,
        "input_token_details": {
            "cache_read": 0
        }
    },
    
    # 3. RESPONSE METADATA (Provider Specific Data)
    "response_metadata": {
        "model_name": "gpt-4o-2024-05-13",
        "system_fingerprint": "fp_aaaa123456",
        "finish_reason": "tool_calls",
        "logprobs": None
    }
}
